# Ribosome Network — Synthetase (miner) on Kaggle GPU

Runs the **miner side** of the Bittensor subnet offline: the same
`Synthetase` mechanism code the live neuron serves, with the GA generator
scaled up by GPU screening.

**Kaggle setup**
1. *Settings → Accelerator*: **GPU T4 x2** or **RTX Pro 6000**
2. *Settings → Internet*: **ON** (pip install + optional git fallback)
3. *Input → Add Input*: upload `ribosome-network.zip` as a Dataset
   (otherwise set `REPO_URL` in cell 2 to your fork and let it clone)

**What you verify here**
- environment (torch/CUDA, device plan for 2×T4 vs RTX Pro 6000)
- oracle throughput (Nussinov vs ViennaRNA wheel)
- **GPU coupled-objective GA screening** — the inverse-mRNA paper's
  coupled search (`fitness = structural − β·instability`, β = 2.0) but with
  a population of thousands instead of 200, screened on GPU and refined by
  the real oracle on top-K
- a full miner epoch over all 32 active targets → commit artifacts


In [ ]:
# --- 0. Environment probe -------------------------------------------------
# Verify the accelerator before anything else. Expected on Kaggle:
#   GPU T4 x2      -> 2 devices, 16 GiB each  (kernels run one device each)
#   RTX Pro 6000   -> 1 device, ~96 GiB       (kernels share, larger batches)
import subprocess, sys, platform, json, time

print("python", sys.version.split()[0], "|", platform.platform())
try:
    nvidia = subprocess.run(
        ["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=20).stdout.strip()
    print("nvidia-smi:", nvidia.replace("\n", " | ") or "(none)")
except FileNotFoundError:
    nvidia = ""
    print("nvidia-smi not found - switch the notebook Accelerator to GPU!")

import torch
n_dev = torch.cuda.device_count()
print(f"torch {torch.__version__} | cuda available: {torch.cuda.is_available()} | devices: {n_dev}")

if torch.cuda.is_available():
    DEVICES = [f"cuda:{i}" for i in range(n_dev)]
    for i in range(n_dev):
        p = torch.cuda.get_device_properties(i)
        print(f"  cuda:{i} -> {p.name}, {p.total_memory/2**30:.1f} GiB")
else:
    DEVICES = ["cpu"]
    print("WARNING: no CUDA - kernels fall back to CPU (slow but correct)")
GPU_MEM_GIB = (torch.cuda.get_device_properties(0).total_memory / 2**30
               if torch.cuda.is_available() else 0.0)
IS_T4 = "T4" in (nvidia or "")
print("device plan:", DEVICES)


In [ ]:
# --- 1. Dependencies ------------------------------------------------------
# Mechanism core needs only numpy; ViennaRNA ships as a pip wheel (folds via
# bundled libRNA, no conda needed on Kaggle). bittensor is NOT needed here:
# these notebooks exercise the same mechanism code path offline that the
# live neurons run on testnet.
import subprocess, sys

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout

sh(f"{sys.executable} -m pip install -q numpy pytest viennarna")

try:
    import ViennaRNA
    print("ViennaRNA wheel OK - physics-grade 2D oracle available")
except ImportError:
    print("ViennaRNA unavailable - notebooks will use Nussinov oracle")

import numpy, pytest
print("numpy", numpy.__version__, "| pytest", pytest.__version__)


In [ ]:
# --- 2. Package bootstrap -------------------------------------------------
# Find the ribosome-network package. Two supported paths:
#   a) Kaggle Dataset: upload ribosome-network.zip (or the folder) as an
#      input ("Add Input" -> your dataset). This cell locates and extracts it.
#   b) git clone: set REPO_URL below to your GitHub repo and run once.
import glob, zipfile, shutil, sys, os
from pathlib import Path

REPO_URL = "https://github.com/RibosomeNetwork/ribosome-network"  # <- your fork
WORK = Path("/kaggle/working")
candidates = (glob.glob("/kaggle/input/**/*ribosome*", recursive=True)
              + glob.glob("/kaggle/input/*/*.zip"))
target = None
for c in candidates:
    if c.endswith(".zip") and "ribosome" in c.lower():
        target = c
        with zipfile.ZipFile(c) as z:
            z.extractall(WORK / "pkg")
        break
if target is None and candidates:
    target = candidates[0]  # a dataset directory

root = None
for base in ([WORK / "pkg"] + [Path(c) for c in candidates]):
    if base is None:
        continue
    for p in [base, *base.glob("**/ribosome")]:
        if p.name == "ribosome" and p.is_dir():
            root = p.parent
            break
    if root:
        break

if root is None:
    print("no Kaggle dataset found - cloning", REPO_URL)
    os.system(f"git clone -q {REPO_URL} {WORK/'ribosome-network'}")
    root = WORK / "ribosome-network"

sys.path.insert(0, str(root))
os.chdir(root)
print("package root:", root)

from ribosome import constants  # noqa: E402
print("mechanism constants: theta_dup=%.2f w_div=%.1f T_rot=%d B=%d K=%d"
      % (constants.THETA_DUP, constants.W_DIV, constants.T_ROT,
         constants.SCORE_REVEAL_DELAY_B, constants.K_CANDIDATES))


In [ ]:
# --- 3. Sanity: run the mechanism test-suite ------------------------------
# 92 tests, ~10 s CPU. If this is green, the Kaggle runtime executes the
# exact code path the testnet neurons use.
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pytest", "tests/", "-q", "--no-header"],
                   capture_output=True, text=True, timeout=600)
print(r.stdout[-1200:])
assert " failed" not in r.stdout.splitlines()[-1], "test-suite failed!"


## Oracle throughput (CPU, sequential DP)

Folding is a sequential dynamic program — it stays on CPU and is
memoized. The GPU enters where the paper's pipeline needs *throughput*:
screening huge candidate populations with the coupled objective
(next cells), which is embarrassingly batch-parallel.


In [ ]:
# --- 4. Oracle benchmark ---------------------------------------------------
import time
from ribosome.oracle import StubOracle
from ribosome.data_targets import load_pool
from ribosome.rna import random_sequence
import random

pool = load_pool()
rng = random.Random(0)
seqs = [random_sequence(rng, 110) for _ in range(40)] +        [t.sequence for t in pool.active]
print(f"benchmark over {len(seqs)} sequences (pool length included)")

def bench(fold_fn, label, n=40):
    t0 = time.perf_counter()
    for s in seqs[:n]:
        fold_fn(s)
    dt = time.perf_counter() - t0
    print(f"{label:16s} {dt/n*1000:7.2f} ms/seq   ({n/dt:8.1f} seq/s)")
    return n / dt

stub = StubOracle()
rates = {"nussinov": bench(stub.fold, "Nussinov")}
try:
    from ribosome.oracle import PyViennaRNAOracle
    vr = PyViennaRNAOracle()
    rates["viennarna"] = bench(vr.fold, "ViennaRNA")
except ImportError:
    print("ViennaRNA wheel unavailable - skipped")

import json
json.dump(rates, open("oracle_rates.json", "w"), indent=2)


## GPU coupled-objective screening (β = 2.0, multi-GPU)

The coupled objective from *Stability-Aware mRNA Inverse Design*:

$$fitness(s) = \underbrace{compat(s, T)}_{\text{structural}} - \beta \cdot \underbrace{instab(s)}_{\text{stability}}$$

GPU kernel (vectorized over a batch of sequences):
- `compat` = fraction of the target's base pairs that the candidate realizes
  as Watson-Crick / wobble (AU, GC, GU) — a differentiable-free proxy for
  base-pair recovery
- `instab` = weak-pair fraction + unpaired-A/U penalty (same shape as the
  paper's instability surrogate)

Population 4096 (paper: 200). Split across all GPUs. Top candidates are then
**refined with the real oracle** (Nussinov/ViennaRNA) — screen wide, verify
narrow, exactly the forward-model-guided search pattern.


In [ ]:
# --- 5. GPU fitness kernel --------------------------------------------------
import torch

BASES = "AUGC"
CODE = {c: i for i, c in enumerate(BASES)}

# wobble + Watson-Crick compatibility over (i, j) target pairs
def _compat_code(a: int, b: int) -> float:
    pair = {BASES[a], BASES[b]}
    if pair in ({"A", "U"}, {"G", "C"}, {"G", "U"}):
        return 1.0
    return 0.0

COMPAT = torch.tensor(
    [[_compat_code(a, b) for b in range(4)] for a in range(4)],
    dtype=torch.float32,
)  # [4, 4] lookup

def make_target_layout(target):
    """Precompute target pair/unpaired index tensors (CPU, once)."""
    from ribosome.rna import parse_dot_bracket
    pairs = parse_dot_bracket(target.dot_bracket)
    paired = {i for p in pairs for i in p}
    unpaired = torch.tensor([i for i in range(target.length)
                             if i not in paired], dtype=torch.long)
    pi = torch.tensor([p[0] for p in pairs], dtype=torch.long)
    pj = torch.tensor([p[1] for p in pairs], dtype=torch.long)
    return pi, pj, unpaired

def gpu_fitness(batch_seqs: torch.Tensor, layout, beta: float,
                device: str) -> torch.Tensor:
    """batch_seqs: int64 [B, L]; returns fitness [B]."""
    pi, pj, unpaired = layout
    s = batch_seqs.to(device)
    if len(pi):
        a, b = s[:, pi.to(device)], s[:, pj.to(device)]
        compat = COMPAT.to(device)[a, b].mean(dim=1)
    else:
        compat = torch.zeros(s.shape[0], device=device)
    n = s.shape[1]
    if len(unpaired):
        u = s[:, unpaired.to(device)]
        unpaired_au = ((u == 0) | (u == 3)).float().mean(dim=1)
    else:
        unpaired_au = torch.zeros(s.shape[0], device=device)
    # instability proxy: compatible-pair deficit + unpaired A/U exposure
    instab = (1.0 - compat) * 0.6 + unpaired_au * 0.4
    return compat - beta * instab

def encode(seqs):
    import numpy as np
    arr = np.zeros((len(seqs), max(len(s) for s in seqs)), dtype=np.int64)
    for r, s in enumerate(seqs):
        for c, ch in enumerate(s.upper().replace("T", "U")):
            arr[r, c] = CODE[ch]
    return torch.from_numpy(arr)

def multi_gpu_fitness(seqs, layout, beta=2.0):
    """Score a python list of sequences on every available device."""
    t = encode(seqs)
    outs = []
    chunk = (t.shape[0] + len(DEVICES) - 1) // len(DEVICES)
    for d, dev in enumerate(DEVICES):
        piece = t[d * chunk:(d + 1) * chunk]
        if piece.shape[0] == 0:
            continue
        outs.append(gpu_fitness(piece, layout, beta, dev).cpu())
    return torch.cat(outs).numpy()

# throughput test
from ribosome.rna import random_sequence
big = [random_sequence(__import__("random").Random(1), 110) for _ in range(4096)]
layout = make_target_layout(pool.active[0])
t0 = time.perf_counter()
scores = multi_gpu_fitness(big, layout)
dt = time.perf_counter() - t0
dev_tag = "+".join(DEVICES)
print(f"screened {len(big)} seqs x {len(layout[0])} target pairs "
      f"in {dt*1000:.1f} ms on [{dev_tag}] "
      f"({len(big)/dt:,.0f} seq/s)")
assert len(scores) == len(big)


In [ ]:
# --- 6. GPU-screened GA (coupled search, paper's operator set) --------------
import random
from ribosome.generators.ga import ga_fitness_surrogate

def gpu_ga(target, pop_size=4096, generations=25, beta=2.0,
           mutation=0.08, crossover=0.7, elite=32, seed=0):
    """Coupled GA where ALL fitness evaluation runs on GPU (screen), and the
    top `elite` candidates are re-ranked by the real oracle (verify)."""
    from ribosome.oracle import StubOracle
    rng = random.Random(seed)
    layout = make_target_layout(target)
    oracle = StubOracle()
    L = target.length

    pop = [random_sequence(rng, L) for _ in range(pop_size)]
    history = []
    for gen in range(generations):
        fit = multi_gpu_fitness(pop, layout, beta)
        history.append(float(fit.max()))
        order = fit.argsort()[::-1]
        elites = [pop[i] for i in order[:elite]]
        # tournament selection on GPU scores
        def pick():
            a, b = rng.sample(range(pop_size), 2)
            return pop[a] if fit[a] > fit[b] else pop[b]
        children = elites[:]
        while len(children) < pop_size:
            p1, p2 = pick(), pick()
            if rng.random() < crossover:
                cut = rng.randint(1, L - 1)
                child = p1[:cut] + p2[cut:]
            else:
                child = p1
            child = "".join(
                rng.choice(BASES) if rng.random() < mutation else c
                for c in child)
            children.append(child)
        pop = children
    # oracle refinement of the elite set
    top = [pop[i] for i in multi_gpu_fitness(pop, layout, beta).argsort()[::-1][:elite]]
    refined = sorted(top, key=lambda s: ga_fitness_surrogate(s, target), reverse=True)
    return refined, history

target = pool.active[0]
t0 = time.perf_counter()
best, history = gpu_ga(target, seed=3)
dt = time.perf_counter() - t0
print(f"GPU GA: {4096*25:,} fitness evals in {dt:.1f}s "
      f"({4096*25/dt:,.0f} evals/s) -> best fitness {history[-1]:.4f}")
print("best candidate:", best[:60], "...")

# uplift vs the paper's CPU-config GA (pop 12 x 4 gens) — both judged by the
# SAME ground-truth metric: oracle-verified structure recovery (base-pair F1
# between the folded best candidate and the target)
from ribosome.generators import GAGenerator
from ribosome.scoring import base_pair_f1
from ribosome.oracle import StubOracle as _SO
_o = _SO()

def oracle_f1(cands):
    return max(base_pair_f1(_o.fold(s), target.dot_bracket) for s in cands)

t0 = time.perf_counter()
ref = GAGenerator().generate(target, 4, random.Random(3))
cpu_dt = time.perf_counter() - t0
cpu_f1 = oracle_f1(ref)
gpu_f1 = oracle_f1(best[:4])
print(f"oracle-verified structure F1: CPU-config GA {cpu_f1:.3f} | "
      f"GPU-screened GA {gpu_f1:.3f} -> uplift {gpu_f1-cpu_f1:+.3f} "
      f"(CPU-config search: {cpu_dt*1000:.0f} ms)")

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 3), constrained_layout=True)
ax.plot(history)
ax.set_xlabel("generation"); ax.set_ylabel("best GPU fitness")
ax.set_title(f"GPU-screened GA convergence ({target.id})")
plt.savefig("gpu_ga_convergence.png", dpi=140); plt.show()


In [ ]:
# --- 7. Full miner epoch: 32 targets, K=4 candidates, commit artifacts ------
import hashlib, json
from ribosome.commit import commitment_hash, join_candidates

artifacts = []
t0 = time.perf_counter()
for epoch, target in enumerate(pool.active):
    best, _ = gpu_ga(target, pop_size=1024, generations=8, seed=epoch)
    payload = join_candidates(best[:4])
    salt = f"kaggle-{epoch}"
    artifacts.append({
        "epoch": epoch, "target_id": target.id,
        "commitment": commitment_hash(payload, salt, epoch, target.id),
        "candidates": best[:4],
    })
dt = time.perf_counter() - t0
print(f"produced commitments for {len(artifacts)} targets in {dt:.1f}s "
      f"({dt/len(artifacts)*1000:.0f} ms/target)")

with open("kaggle_commits.jsonl", "w") as f:
    for a in artifacts:
        f.write(json.dumps(a) + "\n")
print("saved kaggle_commits.jsonl ->", len(artifacts), "commits")
print(json.dumps({k: v for k, v in artifacts[0].items() if k != 'candidates'}, indent=2))


## Take-aways

| item | result |
|---|---|
| mechanism tests | green on Kaggle (92/92) |
| oracle | Nussinov ~ms/seq, ViennaRNA wheel physics-grade |
| GPU screening | 4096-population coupled-objective search, multi-GPU split |
| artifact | `kaggle_commits.jsonl` — real commit-reveal payloads for all 32 targets |

The committed payloads hash exactly like the live miner's
(`SHA256(payload‖salt‖epoch‖target)`), so these artifacts can be replayed
through `neurons/validator.py --mode mock --ledger ...` unchanged.

Next: `02_validator_chaperone_gpu.ipynb` — the chaperone side.
